# Temporal Filtering of 3D Pose Predictions on Fit3D

This notebook implements the temporal post-processing stage of the evaluation pipeline. It takes the 3D predictions produced by the inference notebooks and applies three lightweight temporal filters:

- **One Euro Filter**
- **Savitzky--Golay filter**
- **Median filter**

The experiments cover the three models evaluated in this work:

- **TCPFormer**, with RTMPose 2D detections as input and overlapping-window (Hann) inference.
- **PoseMamba-L**, with RTMPose 2D detections as input and overlapping-window (Hann) inference.
- **MVGFormer**, with synchronized multi-view RGB images.

The notebook has three objectives:

1. **Parameter selection (optional):** reproduce the global search that selects one parameter set per model and filter.
2. **Filtered prediction generation:** apply the final parameters to the 224 non-warm-up sequences of every model.
3. **Verification:** check every generated file and reproduce the PA-MPJPE and MPJVE values reported in the paper.

All filtering is performed **after model inference** and the original predictions are never modified. Result tables and figures are produced in the companion evaluation notebook.

## 1. Experimental Workflow

1. Load the 3D predictions of each model and the Fit3D ground truth.
2. Express predictions and ground truth in the **COMMON15** skeleton, in millimetres and root-centred at the pelvis.
3. *(Optional)* Search the filter parameters. For each model and filter, the selected configuration is the one that **minimizes the mean MPJVE** over all 224 sequences, subject to **PA-MPJPE not increasing by more than 0.5 mm** with respect to the original predictions. A single global configuration is selected per model and filter (no per-subject tuning).
4. Apply the final parameters to every sequence and save each filtered version as a new NumPy array.
5. Verify the generated files and compare the resulting metrics with the values reported in the paper.

The expected prediction format is `(T, J, 3)`, with `J = 17` for TCPFormer and PoseMamba-L and `J = 15` for MVGFormer. Sequences are sampled at **50 Hz** and warm-up sequences are excluded.

### Coordinate frames and units

PA-MPJPE is invariant to rotation, but **MPJVE is not**, because it compares velocity vectors. The two families of models therefore require different reference frames:

| Model | Stored units | Prediction frame | Ground truth used for MPJVE |
|---|---|---|---|
| TCPFormer, PoseMamba-L | metres | camera `65906101` | world GT rotated with the camera rotation `R` |
| MVGFormer | millimetres | world | world GT |

Filters are applied to the arrays **in their stored units** (metres for the monocular models, millimetres for MVGFormer). This matters for the One Euro Filter, whose `beta` parameter multiplies a speed expressed in those units.

## 2. Environment and Imports

In [ ]:
# Configure the execution environment and storage location

import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    # Change only this path if the project data are stored elsewhere.
    STORAGE_ROOT = Path("/content/drive/MyDrive")
else:
    # For local execution, place the data under ./data or change this path.
    STORAGE_ROOT = Path.cwd() / "data"

print("Environment:", "Google Colab" if IN_COLAB else "Local")
print("Storage root:", STORAGE_ROOT)

In [ ]:
# Import the libraries required for filtering, evaluation and bookkeeping

import os
import json
import time
import hashlib
import itertools
from functools import lru_cache

import numpy as np
import pandas as pd

from scipy.signal import savgol_filter
from scipy.ndimage import median_filter

from IPython.display import display

## 3. Data Organization and Configuration

### Input predictions

Each model stores one NumPy array per sequence, organized as `<subject>/<exercise>.npy`:

```text
Fit3D_TCPFormer/pred3d_pose2d_rtmpose_overlap_hann_w243_s60/<subject>/<exercise>.npy   # (T, 17, 3), metres
Fit3D_PoseMamba/pred3d_pose2d_rtmpose_overlap_hann_w243_s60/<subject>/<exercise>.npy   # (T, 17, 3), metres
Fit3D_MVGFormer/predictions_3d/<subject>/<exercise>.npy                                    # (T, 15, 3), millimetres
```

For each model a list of candidate directories is provided; the first one that exists is used. The monocular predictions must come from the overlapping-window inference (`overlap_hann_w243_s60`).

### Ground truth and calibration

- Fit3D 3D annotations (`joints3d_25`, in metres) are read from `VIDEOS/<subject>/<exercise>/joints3d_25.json` (or from `Fit3D_2D/<subject>/gt/<exercise>.json`).
- Camera calibration files are searched under `camera_params/<subject>/`. Only the rotation of camera `65906101` is needed, to express the ground truth in the frame of the monocular predictions.

### Output

Filtered predictions, search results and metric tables are written under `Fit3D_temporal_filtering/`, separate from the original model outputs:

```text
Fit3D_temporal_filtering/
├── filtered_predictions/<model>/<filter and parameters>/<subject>/<exercise>.npy
├── parameter_search/        # raw search results, checkpoints, selected parameters
└── metrics/                 # per-sequence metrics used by the evaluation notebook
```

In [ ]:
# Global configuration

FPS = 50.0

SUBJECTS = ["s03", "s04", "s05", "s07", "s08", "s09", "s10", "s11"]

# 8 subjects x 28 non-warm-up exercises
EXPECTED_SEQUENCES_PER_MODEL = 224

MODEL_ORDER = ["TCPFormer", "PoseMamba-L", "MVGFormer"]

# Camera used for all monocular experiments (frontal view).
MONO_CAMERA_ID = "65906101"

# Dataset and calibration
GT_ROOT = STORAGE_ROOT / "Fit3D_2D"
VIDEOS_ROOT = STORAGE_ROOT / "VIDEOS"
CAMERA_ROOT = STORAGE_ROOT / "camera_params"

# Outputs
OUTPUT_ROOT = STORAGE_ROOT / "Fit3D_temporal_filtering"
FILTERED_ROOT = OUTPUT_ROOT / "filtered_predictions"
SEARCH_ROOT = OUTPUT_ROOT / "parameter_search"
METRICS_ROOT = OUTPUT_ROOT / "metrics"

for folder in [FILTERED_ROOT, SEARCH_ROOT, METRICS_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

# Per-model settings
#   unit_scale : factor that converts the stored array to millimetres
#   gt_swap    : ground-truth side swap needed to match the model's joint order
#   gt_frame   : reference frame in which MPJVE is computed
MODEL_CONFIGS = {
    "TCPFormer": {
        "candidates": [
            STORAGE_ROOT / "Fit3D_TCPFormer"
            / "pred3d_pose2d_rtmpose_overlap_hann_w243_s60",
            STORAGE_ROOT / "Fit3D_TCPFormer_TFG"
            / "pred3d_pose2d_rtmpose_overlap_hann_w243_s60",
        ],
        "num_joints": 17,
        "unit_scale": 1000.0,
        "gt_swap": "legs",
        "gt_frame": "camera",
    },
    "PoseMamba-L": {
        "candidates": [
            STORAGE_ROOT / "Fit3D_PoseMamba"
            / "pred3d_pose2d_rtmpose_overlap_hann_w243_s60",
            STORAGE_ROOT / "Fit3D_PoseMamba_TFG"
            / "pred3d_pose2d_rtmpose_overlap_hann_w243_s60",
            STORAGE_ROOT / "Fit3D_PoseMamba_L_TFG"
            / "pred3d_pose2d_rtmpose_overlap_hann_w243_s60",
        ],
        "num_joints": 17,
        "unit_scale": 1000.0,
        "gt_swap": "legs",
        "gt_frame": "camera",
    },
    "MVGFormer": {
        "candidates": [
            STORAGE_ROOT / "Fit3D_MVGFormer" / "predictions_3d",
            STORAGE_ROOT / "Fit3D_MVGFormer_TFG" / "predictions_3d",
            STORAGE_ROOT / "MVGFormer_results_TFG" / "predictions_3d",
        ],
        "num_joints": 15,
        "unit_scale": 1.0,
        "gt_swap": "arms",
        "gt_frame": "world",
    },
}


def first_existing(paths):
    for path in paths:
        if Path(path).is_dir():
            return Path(path)
    return None


for model_name, cfg in MODEL_CONFIGS.items():
    cfg["prediction_root"] = first_existing(cfg["candidates"])

# ------------------------------------------------------------
# Final filter parameters (those reported in the paper)
# ------------------------------------------------------------
FINAL_FILTER_PARAMS = {
    "TCPFormer": {
        "one_euro": {"freq": 50.0, "min_cutoff": 10.0, "beta": 0.01, "d_cutoff": 2.0},
        "savgol":   {"window_length": 21, "polyorder": 4},
        "median":   {"window_size": 5},
    },
    "PoseMamba-L": {
        "one_euro": {"freq": 50.0, "min_cutoff": 100.0, "beta": 0.01, "d_cutoff": 1.0},
        "savgol":   {"window_length": 11, "polyorder": 4},
        "median":   {"window_size": 3},
    },
    "MVGFormer": {
        "one_euro": {"freq": 50.0, "min_cutoff": 1.0, "beta": 0.001, "d_cutoff": 1.0},
        "savgol":   {"window_length": 21, "polyorder": 2},
        "median":   {"window_size": 41},
    },
}

FILTER_KEYS = ["one_euro", "savgol", "median"]

FILTER_LABELS = {
    "original": "Original",
    "one_euro": "One Euro filter",
    "savgol": "Savitzky--Golay filter",
    "median": "Median filter",
}

# Spatial constraint of the parameter search (mm of PA-MPJPE)
MAX_PA_INCREASE_MM = 0.5

# Run the (long) parameter search? If False, the final parameters above are used directly.
RUN_PARAMETER_SEARCH = False

# Overwrite filtered files that already exist?
OVERWRITE = False

print(f"FPS: {FPS:.0f} Hz | subjects: {len(SUBJECTS)} | expected sequences per model: {EXPECTED_SEQUENCES_PER_MODEL}")
for model_name in MODEL_ORDER:
    root = MODEL_CONFIGS[model_name]["prediction_root"]
    print(f"{model_name:12s} -> {root}")
print("Ground truth :", GT_ROOT, "| exists:", GT_ROOT.is_dir())
print("Calibration  :", CAMERA_ROOT, "| exists:", CAMERA_ROOT.is_dir())

missing = [m for m in MODEL_ORDER if MODEL_CONFIGS[m]["prediction_root"] is None]
if missing:
    raise FileNotFoundError(f"No prediction directory found for: {missing}")

## 4. Temporal Filtering Methods

All filters operate on a pose sequence of shape `(T, J, 3)`. Filtering is performed independently for every joint coordinate and exclusively along the temporal dimension, so no information is mixed between joints or axes.

### One Euro Filter

An adaptive first-order low-pass filter whose cutoff frequency grows with the estimated speed of the signal:

- `freq`: sampling frequency ($f_s$);
- `min_cutoff`: minimum cutoff frequency ($f_{c,\min}$);
- `beta`: coefficient that couples the cutoff to the speed ($\beta$);
- `d_cutoff`: cutoff frequency used to smooth the speed estimate ($f_d$).

The implementation is causal and keeps the first frame unchanged. It is vectorized across several parameter configurations so that the search can evaluate many candidates in a single pass.

### Savitzky--Golay filter

Local least-squares polynomial fit over a centred window (`window_length`, `polyorder`), using polynomial interpolation at the sequence boundaries.

### Median filter

Replaces every value by the median of a centred window of `window_size` frames. The kernel has shape `(N, 1, 1)` so that it only acts along time.

For sequences shorter than the requested window, the window is reduced to the largest valid odd length.

In [ ]:
# Temporal filters

def one_euro_batch(poses, configs):
    '''
    Apply the One Euro Filter with several parameter sets at once.

    poses   : (T, J, 3) array, in its stored units
    configs : list of dicts with keys freq, min_cutoff, beta, d_cutoff
    returns : (N, T, J, 3) float32 array, one filtered sequence per configuration
    '''
    x = np.asarray(poses, dtype=np.float64)
    num_frames, num_joints, num_coords = x.shape
    num_configs = len(configs)

    out = np.empty((num_configs, num_frames, num_joints, num_coords), dtype=np.float32)

    previous_x = np.broadcast_to(x[0], (num_configs, num_joints, num_coords)).copy()
    previous_y = previous_x.copy()
    dx_hat = np.zeros_like(previous_x)
    out[:, 0] = previous_y

    def column(key):
        return np.asarray([p[key] for p in configs], dtype=np.float64)[:, None, None]

    freq = column("freq")
    cutoff_min = column("min_cutoff")
    beta = column("beta")
    d_cutoff = column("d_cutoff")

    alpha_d = 1.0 / (1.0 + freq / (2.0 * np.pi * d_cutoff))

    for t in range(1, num_frames):
        current_x = x[t]

        dx = (current_x - previous_x) * freq
        previous_x = np.broadcast_to(current_x, (num_configs, num_joints, num_coords))

        dx_hat = alpha_d * dx + (1.0 - alpha_d) * dx_hat

        cutoff = cutoff_min + beta * np.abs(dx_hat)
        alpha = 1.0 / (1.0 + freq / (2.0 * np.pi * cutoff))

        previous_y = alpha * current_x + (1.0 - alpha) * previous_y
        out[:, t] = previous_y

    return out


def apply_filter(poses, family, params):
    '''Apply one temporal filter ("one_euro", "savgol" or "median") to a (T, J, 3) sequence.'''
    x = np.asarray(poses, dtype=np.float32)

    if family == "one_euro":
        return one_euro_batch(x, [params])[0]

    if family == "savgol":
        window = min(int(params["window_length"]), len(x))
        if window % 2 == 0:
            window -= 1
        polyorder = int(params["polyorder"])
        if window <= polyorder:
            return x.copy()
        return savgol_filter(
            x, window_length=window, polyorder=polyorder, axis=0, mode="interp"
        ).astype(np.float32)

    if family == "median":
        window = min(int(params["window_size"]), len(x))
        if window % 2 == 0:
            window -= 1
        if window <= 1:
            return x.copy()
        return median_filter(x, size=(window, 1, 1), mode="nearest").astype(np.float32)

    raise ValueError(f"Unknown filter family: {family}")


def parameter_tag(value):
    '''Filesystem-friendly representation of a parameter value.'''
    if isinstance(value, float):
        value = f"{value:g}"
    return str(value).replace(".", "p")


def filter_folder_name(family, params):
    '''Output folder name that encodes the filter parameters.'''
    if family == "one_euro":
        return (
            f"one_euro_freq{parameter_tag(params['freq'])}"
            f"_min{parameter_tag(params['min_cutoff'])}"
            f"_beta{parameter_tag(params['beta'])}"
            f"_d{parameter_tag(params['d_cutoff'])}"
        )
    if family == "savgol":
        return f"savgol_w{params['window_length']}_p{params['polyorder']}"
    if family == "median":
        return f"median_w{params['window_size']}"
    raise ValueError(family)

## 5. Ground Truth and Evaluation Metrics

Both metrics are computed on the **COMMON15** skeleton, expressed in millimetres and root-centred at the pelvis:

- **PA-MPJPE**: mean joint error after a per-frame similarity (Procrustes) alignment. It measures spatial accuracy and is invariant to rotation.
- **MPJVE**: mean norm of the difference between predicted and ground-truth frame-to-frame displacements of each joint (mm/frame). No Procrustes alignment and no scale normalization are applied, so it depends on the coordinate frame (see Section 1).

The COMMON15 skeleton keeps Pelvis, RHip, RKnee, RAnkle, LHip, LKnee, LAnkle, Neck, Nose, LShoulder, LElbow, LWrist, RShoulder, RElbow and RWrist. It is obtained from the Fit3D `joints3d_25` annotations by selecting the indices below, and from the 17-joint monocular output with the same indices (Spine and Head are discarded).

The ground truth requires a left/right correction to match the joint order of each model: legs are swapped for the monocular models and arms for MVGFormer.

In [ ]:
# Ground-truth loading, COMMON15 conversion and metrics

COMMON15_INDICES = [0, 1, 2, 3, 4, 5, 6, 8, 9, 11, 12, 13, 14, 15, 16]

COMMON15_NAMES = [
    "Pelvis", "RHip", "RKnee", "RAnkle", "LHip", "LKnee", "LAnkle",
    "Neck", "Nose", "LShoulder", "LElbow", "LWrist",
    "RShoulder", "RElbow", "RWrist",
]

LEG_SWAP = ([1, 2, 3, 4, 5, 6], [4, 5, 6, 1, 2, 3])
ARM_SWAP = ([9, 10, 11, 12, 13, 14], [12, 13, 14, 9, 10, 11])


def is_warmup(name):
    return str(name).lower().startswith("warmup")


def root_align(x):
    x = np.asarray(x)
    return x - x[:, 0:1, :]


def find_gt_json(subject, video):
    candidates = [
        GT_ROOT / subject / "gt" / f"{video}.json",
        GT_ROOT / subject / "gt (1)" / f"{video}.json",
        VIDEOS_ROOT / subject / video / "joints3d_25.json",
    ]
    for path in candidates:
        if path.is_file():
            return path
    raise FileNotFoundError(
        f"Ground truth not found for {subject}/{video} under {GT_ROOT} or {VIDEOS_ROOT}"
    )


def find_camera_json(subject, video, camera_id=MONO_CAMERA_ID):
    base = CAMERA_ROOT / subject
    candidates = [
        base / camera_id / f"{video}.json",
        base / camera_id / f"{video} (1).json",
        base / "videos" / camera_id / f"{video}.json",
        base / "videos" / camera_id / f"{video} (1).json",
    ]
    for path in candidates:
        if path.is_file():
            return path
    if base.is_dir():
        for path in sorted(base.rglob("*.json")):
            if camera_id.lower() in str(path).lower() and video.lower() in path.stem.lower():
                return path
    raise FileNotFoundError(f"Calibration not found for {subject}/{camera_id}/{video} under {CAMERA_ROOT}")


@lru_cache(maxsize=None)
def camera_rotation(subject, video):
    '''Rotation R of the monocular camera, with X_cam = R @ X_world + T.'''
    path = find_camera_json(subject, video)
    with open(path, "r", encoding="utf-8") as file:
        data = json.load(file)

    rotation = np.asarray(data["extrinsics"]["R"], dtype=np.float64).reshape(3, 3)

    if not np.isfinite(rotation).all() or not np.allclose(rotation @ rotation.T, np.eye(3), atol=0.02):
        raise ValueError(f"Invalid rotation matrix in {path}")

    return rotation


def load_gt_common15(subject, video, gt_swap):
    '''Fit3D ground truth as COMMON15, in mm, side-corrected and pelvis-centred (world frame).'''
    with open(find_gt_json(subject, video), "r", encoding="utf-8") as file:
        data = json.load(file)

    gt = np.asarray(data["joints3d_25"], dtype=np.float32)
    if gt.ndim == 4 and gt.shape[1] == 1:
        gt = gt[:, 0]
    if gt.ndim != 3 or gt.shape[1:] != (25, 3):
        raise ValueError(f"Unexpected ground-truth shape for {subject}/{video}: {gt.shape}")

    gt = gt[:, COMMON15_INDICES, :] * np.float32(1000.0)

    src, dst = LEG_SWAP if gt_swap == "legs" else ARM_SWAP
    gt[:, src, :] = gt[:, dst, :].copy()

    return root_align(gt).astype(np.float32)


def to_common15_mm(pose, cfg):
    '''Convert a stored prediction (or a filtered version of it) to COMMON15, mm, pelvis-centred.'''
    pose = np.asarray(pose, dtype=np.float32)
    if cfg["num_joints"] == 17:
        pose = pose[:, COMMON15_INDICES, :]
    return root_align(pose * np.float32(cfg["unit_scale"])).astype(np.float32)


def pa_mpjpe(pred, gt, chunk=512):
    '''Mean PA-MPJPE (mm) over a sequence; frame-wise similarity alignment (batched SVD).'''
    if pred.shape != gt.shape:
        raise ValueError(f"Shape mismatch: {pred.shape} vs {gt.shape}")

    total = 0.0
    num_frames, num_joints, _ = pred.shape

    for start in range(0, num_frames, chunk):
        stop = min(start + chunk, num_frames)
        X = pred[start:stop].astype(np.float64)
        Y = gt[start:stop].astype(np.float64)

        mx = X.mean(axis=1, keepdims=True)
        my = Y.mean(axis=1, keepdims=True)
        X0, Y0 = X - mx, Y - my
        nx = np.linalg.norm(X0, axis=(1, 2))
        ny = np.linalg.norm(Y0, axis=(1, 2))

        valid = (nx > 1e-8) & (ny > 1e-8)
        aligned = X.copy()

        if valid.any():
            A = X0[valid] / nx[valid, None, None]
            B = Y0[valid] / ny[valid, None, None]
            U, s, Vt = np.linalg.svd(A.transpose(0, 2, 1) @ B)
            R = Vt.transpose(0, 2, 1) @ U.transpose(0, 2, 1)

            # Reflection correction. As in the evaluation of notebooks 01 to 03, the scale uses the sum of the singular
            # values without changing the sign of the smallest one (the Original MVGFormer rows of the arXiv version
            # use the variant that also changes that sign; see the README).
            flip = np.linalg.det(R) < 0
            if flip.any():
                Vt[flip, -1, :] *= -1
                R[flip] = Vt[flip].transpose(0, 2, 1) @ U[flip].transpose(0, 2, 1)

            scale = ny[valid] / nx[valid] * s.sum(axis=1)
            translation = my[valid, 0] - scale[:, None] * np.einsum("fij,fj->fi", R, mx[valid, 0])
            aligned[valid] = scale[:, None, None] * (X[valid] @ R.transpose(0, 2, 1)) + translation[:, None, :]

        total += np.linalg.norm(aligned - Y, axis=-1).sum(dtype=np.float64)

    return float(total / (num_frames * num_joints))


def mpjve(pred, gt_velocity_frame):
    '''MPJVE (mm/frame) between a pelvis-centred prediction and a pelvis-centred GT in the same frame.'''
    if pred.shape != gt_velocity_frame.shape:
        raise ValueError(f"Shape mismatch: {pred.shape} vs {gt_velocity_frame.shape}")

    v_pred = np.diff(pred.astype(np.float64), axis=0)
    v_gt = np.diff(gt_velocity_frame.astype(np.float64), axis=0)
    return float(np.linalg.norm(v_pred - v_gt, axis=-1).mean())


def mpjve_many(filtered_stack, gt_velocity_frame, unit_scale):
    '''MPJVE of many candidate sequences at once; filtered_stack has shape (N, T, 15 or 17, 3).'''
    n_frames = len(gt_velocity_frame)
    pred = filtered_stack[:, :n_frames].astype(np.float64) * unit_scale
    if pred.shape[2] == 17:
        pred = pred[:, :, COMMON15_INDICES, :]
    pred = pred - pred[:, :, 0:1, :]

    v_pred = np.diff(pred, axis=1)
    v_gt = np.diff(gt_velocity_frame.astype(np.float64), axis=0)
    return np.linalg.norm(v_pred - v_gt[None], axis=-1).mean(axis=(1, 2))

In [ ]:
# Load the sequences of one model together with its ground truth and baseline metrics

def list_sequences(model_name):
    root = MODEL_CONFIGS[model_name]["prediction_root"]
    items = []
    for subject in SUBJECTS:
        subject_dir = root / subject
        if not subject_dir.is_dir():
            raise FileNotFoundError(f"Missing directory: {subject_dir}")
        for name in sorted(os.listdir(subject_dir)):
            if name.endswith(".npy") and not is_warmup(name[:-4]):
                items.append((subject, name[:-4]))
    return items


def load_prediction(model_name, subject, video):
    cfg = MODEL_CONFIGS[model_name]
    path = cfg["prediction_root"] / subject / f"{video}.npy"
    pose = np.load(path, allow_pickle=False)

    if pose.ndim != 3 or pose.shape[1:] != (cfg["num_joints"], 3):
        raise ValueError(f"{path}: expected (T, {cfg['num_joints']}, 3), received {pose.shape}")
    if len(pose) == 0 or not np.isfinite(pose).all():
        raise ValueError(f"{path}: empty or non-finite prediction")

    return pose.astype(np.float32), path


def build_case(model_name, subject, video, pose):
    '''Prepare ground truth (spatial and velocity frames) for one sequence.'''
    cfg = MODEL_CONFIGS[model_name]

    gt = load_gt_common15(subject, video, cfg["gt_swap"])
    n_frames = min(len(pose), len(gt))
    gt = gt[:n_frames]

    if cfg["gt_frame"] == "camera":
        # X_cam = R @ X_world + T. The translation cancels after pelvis centring;
        # the rotation must be kept because MPJVE compares velocity vectors.
        gt_velocity_frame = (gt.astype(np.float64) @ camera_rotation(subject, video).T).astype(np.float32)
    else:
        gt_velocity_frame = gt

    return {
        "subject": subject,
        "video": video,
        "n_frames": n_frames,
        "gt": gt,
        "gt_velocity_frame": gt_velocity_frame,
    }


def evaluate_pose(model_name, case, pose):
    '''PA-MPJPE and MPJVE of a (possibly filtered) stored-units prediction.'''
    cfg = MODEL_CONFIGS[model_name]
    pred = to_common15_mm(pose[: case["n_frames"]], cfg)
    return pa_mpjpe(pred, case["gt"]), mpjve(pred, case["gt_velocity_frame"])


def load_model_cases(model_name, verbose=True):
    sequences = list_sequences(model_name)

    if len(sequences) != EXPECTED_SEQUENCES_PER_MODEL:
        raise RuntimeError(
            f"{model_name}: expected {EXPECTED_SEQUENCES_PER_MODEL} sequences, found {len(sequences)}"
        )

    cases = []
    for index, (subject, video) in enumerate(sequences, start=1):
        pose, path = load_prediction(model_name, subject, video)
        case = build_case(model_name, subject, video, pose)
        case["pose"] = pose
        case["path"] = path
        case["baseline_pa"], case["baseline_mpjve"] = evaluate_pose(model_name, case, pose)
        cases.append(case)

        if verbose and (index % 56 == 0 or index == len(sequences)):
            print(f"  {model_name}: {index}/{len(sequences)} sequences loaded")

    return cases

## 6. Parameter Search (optional)

The search reproduces the selection of the filter parameters. It is disabled by default (`RUN_PARAMETER_SEARCH = False`) because it takes a long time; the final parameters used in the paper are stored in `FINAL_FILTER_PARAMS`.

### Selection rule

For each model and filter, using all 224 sequences and one global parameter set:

1. Compute the mean MPJVE of every candidate configuration.
2. Sort the candidates by MPJVE (ascending).
3. Go through the sorted list and keep the first candidate whose mean PA-MPJPE is at most `PA_original + 0.5 mm`.

PA-MPJPE is only evaluated for candidates in order of MPJVE, which avoids computing it for the full grid. Progress is checkpointed so that an interrupted search can be resumed.

### Search grids

| Filter | Monocular models | MVGFormer |
|---|---|---|
| One Euro | `min_cutoff` ∈ {0.5 … 100}, `beta` ∈ {0 … 0.01}, `d_cutoff` ∈ {0.1, 0.5, 1, 2} | union of the grids explored for MVGFormer (`min_cutoff` 0.5 → 100, `beta` 0.001 → 1, `d_cutoff` 0.005 → 2) |
| Savitzky--Golay | window ∈ {5 … 51}, polyorder ∈ {2 … 5} | window ∈ {5 … 21}, polyorder ∈ {2 … 5} |
| Median | window ∈ {3 … 41} | window ∈ {3 … 41} |

The wider `beta` range explored for MVGFormer reflects its much noisier, frame-wise predictions.

In [ ]:
# Search grids


def grid_to_list(grid):
    keys = list(grid.keys())
    return [dict(zip(keys, values)) for values in itertools.product(*[grid[k] for k in keys])]


def unique_configs(configs):
    unique = {}
    for config in configs:
        unique[json.dumps(config, sort_keys=True)] = config
    return list(unique.values())


# ---- Monocular models (TCPFormer, PoseMamba-L) ----
MONO_ONE_EURO_GRID = {
    "freq": [FPS],
    "min_cutoff": [0.5, 1.0, 1.5, 2.0, 3.0, 3.3333333333, 4.0, 5.0, 7.5, 10.0, 20.0, 50.0, 100.0],
    "beta": [0.0, 0.0001, 0.0005, 0.001, 0.003, 0.005, 0.01],
    "d_cutoff": [0.1, 0.5, 1.0, 2.0],
}
MONO_SAVGOL_GRID = {
    "window_length": [5, 7, 9, 11, 15, 17, 21, 31, 41, 51],
    "polyorder": [2, 3, 4, 5],
}

# ---- MVGFormer: union of the grids explored for this model ----
MVG_ONE_EURO_GRIDS = [
    {   # base grid
        "freq": [FPS],
        "min_cutoff": [0.5, 0.8, 0.8333333333, 1.0, 1.2, 1.3333333333, 1.5, 1.6666666667,
                       2.0, 2.5, 3.3333333333, 4.0, 5.0],
        "beta": [0.001, 0.003, 0.005, 0.01, 0.02, 0.03, 0.05],
        "d_cutoff": [1.0, 1.6666666667, 2.0],
    },
    {   # refined
        "freq": [FPS],
        "min_cutoff": [4.0, 5.0, 6.0, 7.0, 8.0],
        "beta": [0.03, 0.05, 0.075, 0.10],
        "d_cutoff": [0.5, 0.75, 1.0, 1.5, 1.6666666667],
    },
    {   # final
        "freq": [FPS],
        "min_cutoff": [8.0, 10.0, 12.0],
        "beta": [0.075, 0.10, 0.15, 0.20],
        "d_cutoff": [0.25, 0.50, 0.75],
    },
    {   # broad
        "freq": [FPS],
        "min_cutoff": [4.0, 8.0, 12.0, 20.0, 35.0, 60.0, 100.0],
        "beta": [0.01, 0.03, 0.075, 0.15, 0.30, 0.60, 1.00],
        "d_cutoff": [0.01, 0.03, 0.10, 0.25, 0.50, 1.00, 2.00],
    },
    {   # local
        "freq": [FPS],
        "min_cutoff": [12.0, 16.0, 20.0, 25.0, 32.0],
        "beta": [0.005, 0.0075, 0.01, 0.0125, 0.015],
        "d_cutoff": [0.005, 0.0075, 0.01, 0.0125, 0.015],
    },
]
MVG_SAVGOL_GRID = {
    "window_length": [5, 7, 9, 11, 15, 17, 21],
    "polyorder": [2, 3, 4, 5],
}

MEDIAN_WINDOWS = [3, 5, 7, 9, 11, 15, 21, 31, 41]


def search_space(model_name, family):
    '''List of candidate parameter dictionaries for one model and filter.'''
    is_mvg = model_name == "MVGFormer"

    if family == "one_euro":
        grids = MVG_ONE_EURO_GRIDS if is_mvg else [MONO_ONE_EURO_GRID]
        configs = [c for grid in grids for c in grid_to_list(grid)]
    elif family == "savgol":
        grid = MVG_SAVGOL_GRID if is_mvg else MONO_SAVGOL_GRID
        configs = [c for c in grid_to_list(grid) if c["window_length"] > c["polyorder"]]
    elif family == "median":
        configs = [{"window_size": w} for w in MEDIAN_WINDOWS]
    else:
        raise ValueError(family)

    return unique_configs(configs)


for _model in MODEL_ORDER:
    print(_model, {family: len(search_space(_model, family)) for family in FILTER_KEYS})

In [ ]:
# Search functions

ONE_EURO_BATCH = 16
SAVE_CHECKPOINT_EVERY = 10


def _signature(model_name, family, configs, cases):
    payload = {
        "protocol": "COMMON15_mpjve_search_v1",
        "model": model_name,
        "filter": family,
        "configs": configs,
        "cases": [(c["subject"], c["video"], len(c["pose"]), c["n_frames"]) for c in cases],
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()


def search_mpjve(model_name, family, configs, cases):
    '''Mean MPJVE of every configuration over all cases (resumable).'''
    cfg = MODEL_CONFIGS[model_name]
    total = len(cases)
    signature = _signature(model_name, family, configs, cases)
    checkpoint = SEARCH_ROOT / "checkpoints" / f"{model_name}_{family}_progress.npz"
    checkpoint.parent.mkdir(parents=True, exist_ok=True)

    done, sums = 0, np.zeros(len(configs), dtype=np.float64)
    if checkpoint.is_file():
        with np.load(checkpoint, allow_pickle=False) as saved:
            if str(saved["signature"].item()) == signature:
                done, sums = int(saved["done"]), saved["sums"].astype(np.float64)
                print(f"  Resuming from {done}/{total} sequences")

    start = time.monotonic()
    for index in range(done, total):
        case = cases[index]
        pose = case["pose"]

        if family == "one_euro":
            for a in range(0, len(configs), ONE_EURO_BATCH):
                b = min(a + ONE_EURO_BATCH, len(configs))
                stack = one_euro_batch(pose, configs[a:b])
                sums[a:b] += mpjve_many(stack, case["gt_velocity_frame"], cfg["unit_scale"])
        else:
            for k, params in enumerate(configs):
                filtered = apply_filter(pose, family, params)
                sums[k] += mpjve_many(filtered[None], case["gt_velocity_frame"], cfg["unit_scale"])[0]

        processed = index + 1
        if processed % SAVE_CHECKPOINT_EVERY == 0 or processed == total:
            tmp = checkpoint.with_name(checkpoint.stem + ".tmp.npz")
            np.savez_compressed(tmp, signature=np.array(signature), done=np.array(processed), sums=sums)
            os.replace(tmp, checkpoint)

        if processed % 28 == 0 or processed == total:
            elapsed = time.monotonic() - start
            rate = elapsed / max(processed - done, 1)
            print(f"  [{model_name} | {family}] {processed}/{total} sequences "
                  f"| ~{rate * (total - processed) / 60:.1f} min remaining")

    return sums / total


def mean_pa_of_candidate(model_name, family, params, cases):
    scores = []
    for case in cases:
        filtered = apply_filter(case["pose"], family, params)
        scores.append(evaluate_pose(model_name, case, filtered)[0])
    return float(np.mean(scores))


def select_parameters(model_name, family, cases):
    '''Minimum mean MPJVE subject to PA <= PA_original + MAX_PA_INCREASE_MM.'''
    configs = search_space(model_name, family)
    mpjve_scores = search_mpjve(model_name, family, configs, cases)

    baseline_pa = float(np.mean([c["baseline_pa"] for c in cases]))
    pa_scores = {}
    selected = None

    for rank, k in enumerate(np.argsort(mpjve_scores, kind="stable"), start=1):
        k = int(k)
        pa_scores[k] = mean_pa_of_candidate(model_name, family, configs[k], cases)
        if pa_scores[k] <= baseline_pa + MAX_PA_INCREASE_MM:
            selected = k
            break

    if selected is None:
        raise RuntimeError(f"No admissible configuration for {model_name}/{family}")

    rows = [
        {
            "model": model_name,
            "filter": family,
            "config_id": k,
            "params": json.dumps(configs[k], sort_keys=True),
            "MPJVE_mm_frame": float(mpjve_scores[k]),
            "PA_MPJPE_mm": pa_scores.get(k, np.nan),
            "selected": k == selected,
        }
        for k in range(len(configs))
    ]

    print(f"  -> selected after checking {rank} candidate(s): {configs[selected]} "
          f"(MPJVE {mpjve_scores[selected]:.4f}, PA {pa_scores[selected]:.4f} vs original {baseline_pa:.4f})")

    return configs[selected], rows

In [ ]:
# Run the parameter search (only when RUN_PARAMETER_SEARCH is True)

SELECTED_PARAMS = {}

if RUN_PARAMETER_SEARCH:
    search_rows = []

    for model_name in MODEL_ORDER:
        print("=" * 80)
        print(model_name)
        print("=" * 80)

        cases = load_model_cases(model_name)
        SELECTED_PARAMS[model_name] = {}

        for family in FILTER_KEYS:
            print(f"\n{family}: {len(search_space(model_name, family))} candidate configurations")
            params, rows = select_parameters(model_name, family, cases)
            SELECTED_PARAMS[model_name][family] = params
            search_rows.extend(rows)

        pd.DataFrame(search_rows).to_csv(SEARCH_ROOT / "parameter_search_results.csv", index=False)
        (SEARCH_ROOT / "selected_filter_params.json").write_text(
            json.dumps(SELECTED_PARAMS, indent=2), encoding="utf-8"
        )

    # Compare the search outcome with the parameters reported in the paper
    print("\n" + "=" * 80)
    print("SEARCH RESULT vs PARAMETERS IN THE PAPER")
    print("=" * 80)
    all_match = True
    for model_name in MODEL_ORDER:
        for family in FILTER_KEYS:
            found, paper = SELECTED_PARAMS[model_name][family], FINAL_FILTER_PARAMS[model_name][family]
            same = json.dumps(found, sort_keys=True) == json.dumps(paper, sort_keys=True)
            all_match &= same
            print(f"{'OK  ' if same else 'DIFF'} {model_name:12s} {family:9s} search={found} paper={paper}")
    print("\nAll selected parameters match the paper." if all_match else
          "\nSome parameters differ from the paper: review them before regenerating the results.")
else:
    print("Parameter search skipped (RUN_PARAMETER_SEARCH = False). "
          "Using FINAL_FILTER_PARAMS.")

## 7. Final Smoothed Prediction Generation

The parameters stored in `FINAL_FILTER_PARAMS` are applied to the complete set of sequences of every model. For each original prediction three new arrays are written (One Euro, Savitzky--Golay and Median), preserving the `<subject>/<exercise>.npy` layout and the stored units.

Each filter is saved in its own folder whose name encodes the parameters, for example `one_euro_freq50_min10_beta0p01_d2` or `savgol_w21_p4`. Existing files are kept unless `OVERWRITE = True`. A manifest with the input and output path of every file and the parameters used is saved as `filtered_predictions_manifest.csv`.

In [ ]:
# Generate the filtered predictions for all models


def safe_np_save(path, array, retries=5, sleep_sec=0.7):
    '''Save a NumPy array through a temporary file (robust against Drive hiccups).'''
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp.npy")
    last_error = None

    for _ in range(retries):
        try:
            np.save(temporary, array)
            os.replace(temporary, path)
            return
        except Exception as error:  # noqa: BLE001
            last_error = error
            if temporary.exists():
                try:
                    temporary.unlink()
                except OSError:
                    pass
            time.sleep(sleep_sec)

    raise last_error


def filtered_path(model_name, family, subject, video):
    folder = filter_folder_name(family, FINAL_FILTER_PARAMS[model_name][family])
    return FILTERED_ROOT / model_name / folder / subject / f"{video}.npy"


manifest_rows = []
counts = {"saved": 0, "existing": 0}

for model_name in MODEL_ORDER:
    cfg = MODEL_CONFIGS[model_name]
    sequences = list_sequences(model_name)

    if len(sequences) != EXPECTED_SEQUENCES_PER_MODEL:
        raise RuntimeError(f"{model_name}: expected {EXPECTED_SEQUENCES_PER_MODEL} sequences, found {len(sequences)}")

    print(f"\n{model_name}: {len(sequences)} sequences | input: {cfg['prediction_root']}")

    for index, (subject, video) in enumerate(sequences, start=1):
        pose, input_path = load_prediction(model_name, subject, video)

        for family in FILTER_KEYS:
            params = FINAL_FILTER_PARAMS[model_name][family]
            output_path = filtered_path(model_name, family, subject, video)

            if output_path.is_file() and not OVERWRITE:
                counts["existing"] += 1
            else:
                filtered = apply_filter(pose, family, params).astype(np.float32)
                safe_np_save(output_path, filtered)
                counts["saved"] += 1

            manifest_rows.append({
                "model": model_name,
                "filter": family,
                "subject": subject,
                "video": video,
                "input_path": str(input_path),
                "output_path": str(output_path),
                "params": json.dumps(params, sort_keys=True),
            })

        if index % 56 == 0 or index == len(sequences):
            print(f"  {index}/{len(sequences)} sequences processed")

df_manifest = pd.DataFrame(manifest_rows)
df_manifest.to_csv(OUTPUT_ROOT / "filtered_predictions_manifest.csv", index=False)
(OUTPUT_ROOT / "final_filter_params.json").write_text(json.dumps(FINAL_FILTER_PARAMS, indent=2), encoding="utf-8")

print(f"\nFiles written: {counts['saved']} | already existing: {counts['existing']}")
print(f"Manifest: {OUTPUT_ROOT / 'filtered_predictions_manifest.csv'}")

## 8. Verification

The verification has two parts.

**File integrity.** For every model, filter and sequence it checks that:

- the filtered file exists and there are exactly 224 per model and filter;
- shape and number of joints equal those of the original, and the dtype is `float32`;
- all values are finite;
- the filtered array equals a fresh recomputation of the filter from the original prediction;
- the One Euro output keeps the first frame of the original, and every filter actually modifies the sequence.

**Reproduction of the paper.** PA-MPJPE and MPJVE are computed for the original and the three filtered versions of every model, and compared with the values reported in the paper. The per-sequence metrics are saved in `metrics/smoothing_COMMON15_per_video.csv`, which the evaluation notebook uses to build tables and figures.

In [ ]:
# File-integrity verification

errors = []
warnings = []

for model_name in MODEL_ORDER:
    cfg = MODEL_CONFIGS[model_name]
    sequences = list_sequences(model_name)

    for family in FILTER_KEYS:
        params = FINAL_FILTER_PARAMS[model_name][family]
        n_files = 0
        n_modified = 0

        for subject, video in sequences:
            original, _ = load_prediction(model_name, subject, video)
            path = filtered_path(model_name, family, subject, video)

            if not path.is_file():
                errors.append(f"missing: {path}")
                continue

            filtered = np.load(path, allow_pickle=False)
            n_files += 1

            if filtered.shape != original.shape:
                errors.append(f"shape {filtered.shape} != {original.shape}: {path}")
                continue
            if filtered.dtype != np.float32:
                errors.append(f"dtype {filtered.dtype}: {path}")
            if not np.isfinite(filtered).all():
                errors.append(f"non-finite values: {path}")
                continue

            recomputed = apply_filter(original, family, params).astype(np.float32)
            if not np.allclose(filtered, recomputed, atol=1e-6, rtol=0):
                errors.append(f"differs from recomputation: {path}")

            if family == "one_euro" and not np.array_equal(filtered[0], original[0]):
                errors.append(f"One Euro changed the first frame: {path}")

            if not np.allclose(filtered, original, atol=1e-9, rtol=0):
                n_modified += 1

        if n_files != EXPECTED_SEQUENCES_PER_MODEL:
            errors.append(f"{model_name}/{family}: {n_files} files, expected {EXPECTED_SEQUENCES_PER_MODEL}")
        if n_modified == 0:
            warnings.append(f"{model_name}/{family}: the filter did not modify any sequence")

        print(f"{model_name:12s} | {family:9s} | {n_files:3d} files | {n_modified:3d} modified")

print(f"\nErrors: {len(errors)} | warnings: {len(warnings)}")
for message in errors[:20]:
    print("  ERROR  ", message)
for message in warnings[:20]:
    print("  WARNING", message)

if errors:
    raise RuntimeError("File-integrity verification failed.")
print("\nFile-integrity verification passed.")

In [ ]:
# Reproduce the PA-MPJPE and MPJVE values of the paper

# Values reported in the paper (COMMON15, all 224 sequences per model).
PAPER_REFERENCE = {
    # model: {version: (PA-MPJPE mm, MPJVE mm/frame)}
    "TCPFormer": {
        "original": (69.566, 2.943), "one_euro": (69.713, 2.913),
        "savgol": (69.555, 2.874), "median": (69.563, 2.905),
    },
    "PoseMamba-L": {
        "original": (68.768, 2.859), "one_euro": (68.769, 2.859),
        "savgol": (68.767, 2.856), "median": (68.768, 2.858),
    },
    # MVGFormer original: 66.698 with this evaluation; the arXiv version lists 66.691 (computed with a Procrustes
    # variant that also applies the reflection correction to the scale)
    "MVGFormer": {
        "original": (66.698, 26.458), "one_euro": (67.091, 6.550),
        "savgol": (62.102, 4.905), "median": (60.385, 4.192),
    },
}

# Tolerances on the global means
TOLERANCE_PA_MM = 0.02
TOLERANCE_MPJVE = 0.01
TOLERANCE_PA_MM_MVG = 0.05   # MVGFormer: small numerical differences with respect to the original evaluation

per_video_rows = []

for model_name in MODEL_ORDER:
    cases = load_model_cases(model_name, verbose=False)

    for family in ["original"] + FILTER_KEYS:
        for case in cases:
            if family == "original":
                pose = case["pose"]
            else:
                pose = np.load(filtered_path(model_name, family, case["subject"], case["video"]))
            pa, velocity_error = evaluate_pose(model_name, case, pose)

            per_video_rows.append({
                "model": model_name,
                "version": FILTER_LABELS[family],
                "filter": family,
                "subject": case["subject"],
                "video": case["video"],
                "n_eval_frames": case["n_frames"],
                "PA_MPJPE_COMMON15_mm": pa,
                "MPJVE_mm_frame": velocity_error,
            })
    print(f"{model_name}: evaluated")

df_metrics = pd.DataFrame(per_video_rows)
df_metrics.to_csv(METRICS_ROOT / "smoothing_COMMON15_per_video.csv", index=False)

df_global = (
    df_metrics.groupby(["model", "filter"], as_index=False)
    .agg(PA_MPJPE_mm=("PA_MPJPE_COMMON15_mm", "mean"), MPJVE_mm_frame=("MPJVE_mm_frame", "mean"),
         n_sequences=("video", "count"))
)
df_global.to_csv(METRICS_ROOT / "smoothing_COMMON15_global.csv", index=False)

rows, mismatches = [], []
for model_name in MODEL_ORDER:
    for family in ["original"] + FILTER_KEYS:
        row = df_global[(df_global["model"] == model_name) & (df_global["filter"] == family)].iloc[0]
        paper_pa, paper_velocity = PAPER_REFERENCE[model_name][family]
        tolerance_pa = TOLERANCE_PA_MM_MVG if model_name == "MVGFormer" else TOLERANCE_PA_MM

        ok_pa = abs(row["PA_MPJPE_mm"] - paper_pa) <= tolerance_pa
        ok_velocity = abs(row["MPJVE_mm_frame"] - paper_velocity) <= TOLERANCE_MPJVE
        if not (ok_pa and ok_velocity):
            mismatches.append((model_name, family))

        rows.append({
            "model": model_name,
            "version": FILTER_LABELS[family],
            "n": int(row["n_sequences"]),
            "PA computed": round(row["PA_MPJPE_mm"], 3),
            "PA paper": paper_pa,
            "MPJVE computed": round(row["MPJVE_mm_frame"], 3),
            "MPJVE paper": paper_velocity,
            "match": "OK" if (ok_pa and ok_velocity) else "CHECK",
        })

display(pd.DataFrame(rows))

if mismatches:
    print("\nValues that differ from the paper beyond tolerance:", mismatches)
    print("Check the prediction folders (overlap_hann_w243_s60), the parameters and the ground-truth conventions.")
else:
    print("\nAll global PA-MPJPE and MPJVE values reproduce the paper.")

## 9. Generated Outputs

The notebook produces:

- Filtered predictions for TCPFormer, PoseMamba-L and MVGFormer in `filtered_predictions/`, 672 files per model (224 sequences × 3 filters).
- `filtered_predictions_manifest.csv` and `final_filter_params.json`, which document exactly which parameters produced which files.
- `parameter_search/` (only if the search was executed): all evaluated configurations and the selected parameters.
- `metrics/smoothing_COMMON15_per_video.csv` and `smoothing_COMMON15_global.csv`: PA-MPJPE and MPJVE for the original and the filtered predictions, used by the evaluation notebook to build the paper tables and figures.